# 00. Panduan dan pemeriksaan environment

Mulai di sini. Unit penelitian adalah grid-triwulan. Model dan skala fitur dipelajari hanya dari 2021-2025; 2026 merupakan temporal holdout.

[Berikutnya](01_data_collection/01_data_collection.ipynb)

### Daftar bagian

- [1. Inisialisasi dan dependensi](#s1)
- [2. Periksa seluruh pustaka dan interpreter](#s2)
- [3. Rancangan dan urutan eksekusi](#s3)
- [4. Lokasi input dan keluaran](#s4)

<a id="s1"></a>
## 1. Inisialisasi dan dependensi

Jalankan dari atas ke bawah dengan kernel Python 3.12.11 (Seismic Research). Konfigurasi tersentral di config.json.

In [1]:
from pathlib import Path
import os
import sys

os.environ.setdefault("LOKY_MAX_CPU_COUNT", "2")
os.environ.setdefault("OMP_NUM_THREADS", "2")
candidates = []
for parent in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([parent, parent / "seismic_research_2021_2025"])
ROOT = next((p for p in candidates if (p / "config.json").is_file() and (p / "src/common.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Buka notebook dari folder proyek seismic_research_2021_2025.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display, Image, Markdown, FileLink
from src.common import read_config
cfg = read_config(ROOT)
print("Interpreter:", sys.executable)
print("Python:", sys.version.split()[0])
print("Proyek:", ROOT)
print("Training:", cfg["train_start"], "sampai sebelum", cfg["train_end_exclusive"])
print("Testing sampai sebelum (UTC):", cfg["test_end_exclusive"])


Interpreter: C:\Users\yudhi\.local\bin\python3.12.exe
Python: 3.12.11
Proyek: C:\Users\yudhi\Documents\machine-learning\seismic_research_2021_2025
Training: 2021-01-01 sampai sebelum 2026-01-01
Testing sampai sebelum (UTC): 2026-10-26


<a id="s2"></a>
## 2. Periksa seluruh pustaka dan interpreter

Kernel yang disiapkan: **Python 3.12.11 (Seismic Research)**. Pilih nama ini melalui Select Kernel di VS Code. Jika memakai mesin lain, jalankan `setup_kernel.ps1` terlebih dahulu.

In [2]:
import runpy
environment = runpy.run_path(str(ROOT / "scripts/verify_environment.py"))["verify"]()
assert environment["imports"] == "PASS"

{
  "executable": "C:\\Users\\yudhi\\.local\\bin\\python3.12.exe",
  "python_version": "3.12.11 (main, Aug 18 2025, 19:17:54) [MSC v.1944 64 bit (AMD64)]",
  "imports": "PASS",
  "projection": "PASS",
  "packages": {
    "numpy": "2.2.6",
    "pandas": "2.3.3",
    "scipy": "1.18.1",
    "scikit-learn": "1.9.1",
    "matplotlib": "3.11.2",
    "seaborn": "0.13.2",
    "requests": "2.34.2",
    "pyproj": "3.8.0",
    "folium": "0.20.0",
    "branca": "0.8.2",
    "joblib": "1.6.0",
    "threadpoolctl": "3.7.0",
    "ipykernel": "7.4.0",
    "ipython": "9.17.1",
    "nbformat": "5.11.1",
    "nbclient": "0.11.0",
    "jupyterlab": "4.6.4",
    "pytest": "9.1.1",
    "plotly": "6.9.0",
    "geopandas": "1.2.0",
    "shapely": "2.2.0"
  }
}

<a id="s3"></a>
## 3. Rancangan dan urutan eksekusi

Jalankan notebook 01 sampai 10 secara berurutan. Setiap notebook dapat dibuka sendiri dan membaca artefak tahap sebelumnya dari disk. Nomor folder memudahkan pengurutan di Explorer; heading bernomor tampil di Outline/Table of Contents.

In [3]:
import json
display(cfg)
display(Markdown((ROOT / "docs/metodologi.md").read_text(encoding="utf-8")))

{'api_url': 'https://www.seismicportal.eu/fdsnws/event/1/query',
 'train_start': '2021-01-01',
 'train_end_exclusive': '2026-01-01',
 'test_end_exclusive': '2026-10-26',
 'bbox': {'min_lat': -11, 'max_lat': 6, 'min_lon': 95, 'max_lon': 141},
 'grid_km': 50,
 'grid_sensitivity_km': [25, 50, 100],
 'projection': '+proj=laea +lat_0=-2.5 +lon_0=118 +datum=WGS84 +units=m +no_defs',
 'min_magnitude': 3.0,
 'accepted_event_types': ['ke', 'fe', 'de'],
 'magnitude_sensitivity': [2.5, 3.0, 3.5, 4.0],
 'max_depth_km': 800,
 'shallow_depth_km': 70,
 'low_support_count': 5,
 'features': ['log_frequency',
  'avg_magnitude',
  'max_magnitude',
  'avg_depth',
  'shallow_fraction'],
 'k_candidates': [2, 3, 4, 5, 6, 7, 8],
 'dbscan_min_samples': [5, 10, 20],
 'dbscan_distance_quantiles': [0.5, 0.7, 0.85, 0.95],
 'dbscan_max_noise_fraction': 0.5,
 'dbscan_max_largest_cluster_fraction': 0.95,
 'dbscan_min_cluster_size': 20,
 'dbscan_max_clusters': 20,
 'silhouette_sample_size': 2000,
 'silhouette_seeds': 

# Metodologi dan batas pertanggungjawaban

## 1. Tujuan dan pemisahan periode

Tujuan: menemukan profil aktivitas pada unit **grid-triwulan**, mengukur perubahan profil historis, dan memeriksa penerapannya pada periode baru. K-Means dan DBSCAN merupakan unsupervised learning, tanpa label target. Training/model discovery: 2021Q1-2025Q4. Testing temporal: 2026, dengan cutoff **sebelum 26 Oktober 2026 pukul 00:00 UTC**. Q1-Q3 lengkap; Q4 baru observasi sementara. Tidak ada prediksi kapan gempa akan terjadi.

Scaler, centroid, core points, kandidat parameter, k-distance, threshold jarak K-Means, serta kebijakan pemilihan model hanya memakai training. Jangan mengubah parameter setelah melihat testing lalu tetap menyebut testing tersebut sebagai evaluasi independen. Perubahan berbasis testing memerlukan holdout baru.

## 2. Sumber dan kualitas katalog

EMSC/SeismicPortal FDSN Event service, CC BY 4.0. Batas -11..6 lintang dan 95..141 bujur adalah **bounding box**, mencakup laut dan sebagian wilayah tetangga. Hasil tidak mengklaim telah di-clip ke batas negara. Setiap bulan mempunyai cache JSON, waktu pengambilan, URL permintaan, jumlah respons, serta manifest. HTTP error menghentikan collection. Hasil yang mencapai limit dipecah lagi agar tidak terpotong diam-diam. Berhasil mengunduh seluruh interval membuktikan cakupan permintaan, bukan kelengkapan deteksi seismologis.

Depth diambil dari `properties.depth` dalam km. Koordinat ketiga GeoJSON disimpan terpisah; respons nyata memperlihatkan tanda yang berlawanan. Tidak digunakan `abs()` untuk menyembunyikan salah definisi. Depth negatif/di atas 800 km ditandai tidak valid. Rentang itu adalah screening fisik awal yang harus diperiksa untuk outlier, bukan bukti semua data lain benar. Magnitudo -2..10 ditangani sebagai rentang valid awal. ID duplikat mempertahankan lastupdate terbaru; ID hilang memakai kunci atribut. Semua keputusan meninggalkan kolom/CSV audit.

Tipe event asli dipertahankan pada `event_type`. Analisis utama menerima `ke`, `fe`, dan `de` (known/felt/damaging earthquake menurut kode ISF). Tipe tidak dikenal, hilang, atau lainnya masuk tabel rejected dengan alasan `event_type_not_accepted`; tidak otomatis disebut gempa tektonik. Katalog ini juga belum dipisahkan berdasarkan mekanisme sumber.

**M minimum 3.0 masih ambang eksploratif.** Notebook EDA menampilkan distribusi tahunan, author dan jenis magnitudo, jumlah pada beberapa ambang, serta mode histogram 0.1 sebagai diagnostik awal maximum curvature. Mode histogram bukan Mc tervalidasi. Uji sensitivitas M 2.5/3.0/3.5/4.0 tidak menggantikan pengujian completeness yang lebih kuat. Jenis magnitudo dari katalog belum dihomogenisasi menjadi Mw. Perubahan jaringan/author/jenis magnitudo dan kelengkapan pencatatan tetap merupakan kemungkinan pengganggu interpretasi temporal. Sampai isu itu diselesaikan, kesimpulan dibatasi pada **aktivitas yang tercatat dalam katalog**.

## 3. Grid dan agregasi

Grid utama 50 km pada proyeksi **Lambert azimuthal equal-area (LAEA)** berpusat 118 BT, -2.5 LS. Grid origin berada pada kelipatan koordinat proyeksi tetap; tidak bergeser menurut data. Luas sel interior adalah 2.500 km persegi pada bidang equal-area. Panjang sisi di permukaan bumi tidak tepat sama di semua lokasi. Analisis sensitivitas memakai 25 dan 100 km. Polygon ditransformasi kembali ke WGS84 dengan densifikasi sisi.

Universe training: sel yang setidaknya mempunyai satu event di atas ambang pada 2021-2025. Universe testing: gabungan sel historis dan sel baru yang muncul pada 2026. Dengan begitu, penambahan data testing tidak mengubah panel training. Ini bukan seluruh sel kosong dalam bounding box; wilayah yang tidak pernah aktif dalam periode pengamatan tidak diberi profil.

Panel historis melengkapi **20 triwulan untuk setiap grid historis**. Nol event berarti tidak ada event di atas ambang dalam katalog dan interval unduhan yang berhasil, bukan tidak ada gempa secara fisik. Count/frekuensi boleh nol; statistik magnitudo dan depth tetap NaN. `NO_EVENTS`, `INCOMPLETE_FEATURES`, `NOISE`, serta profil cluster dibedakan. Baris untuk clustering harus mempunyai fitur lengkap dan depth coverage >=80%. Proporsi dangkal memakai denominator depth valid. Baris 1-4 event tetap masuk model, namun diberi flag `low_support`; hasilnya perlu kehati-hatian karena maksimum/mean dan proporsi tidak stabil dengan dukungan kecil.

## 4. Fitur dan transformasi

Fitur jarak: log1p(frekuensi per bulan), mean magnitudo, maksimum magnitudo, mean depth, dan proporsi event dangkal (<70 km). Frekuensi triwulan lengkap adalah count/3. Triwulan parsial memakai lama observasi dalam bulan ekuivalen 30.4375 hari. Normalisasi durasi tidak membuat maksimum magnitudo atau proporsi parsial setara dengan periode penuh.

Count, density dan frekuensi mentah tidak sekaligus dimasukkan karena mengulang informasi. Latitude/longitude menjadi identitas spasial, bukan fitur clustering. Mean dan maksimum magnitudo tetap bisa berkorelasi, sehingga tabel korelasi harus ditinjau; log transform dan StandardScaler tidak menghapus semua redundansi. Tidak ada imputasi nol pada statistik kosong. Scaler hanya fit pada training ELIGIBLE dan dipakai transform testing.

## 5. Pemilihan model

K-Means: k=2..8, random_state 42, n_init 20. Pemilihan berdasarkan rata-rata Silhouette dari tiga sampel maksimal 2.000 baris (seed 42/7/21), tie-break DBI lebih rendah dan k lebih kecil. Simpan SD sampling, inertia/elbow, seluruh kandidat dan ukuran cluster. Selisih kecil antarskor tidak membuktikan pemenang universal. Evaluasi internal setelah tuning bersifat eksploratif dan optimistis; tabel testing adalah pemeriksaan tambahan pada data baru.

DBSCAN: min_samples 5/10/20; eps dari kuantil 0.50/0.70/0.85/0.95 jarak tetangga ke-k pada training standar. Query secara eksplisit memasukkan tetangga diri sendiri seperti definisi min_samples DBSCAN. Tidak menetapkan jumlah cluster tepat. Diagnosis training awal menghasilkan kandidat dengan ratusan cluster kecil dan Silhouette tinggi; karena itu kebijakan screening eksploratif menolak fragmentasi berlebihan: 2..20 cluster, anggota cluster terkecil >=20, noise <=50%, dan cluster dominan <=95% assigned. Batas ini **pilihan analitis yang disesuaikan dari diagnosis training**, bukan batas alami seismologi atau praregistrasi penelitian. Tabel sensitivitas kebijakan memperlihatkan konsekuensinya. Jika tak ada kandidat lolos, hasil disimpan dengan status `diagnostic_only`, tidak dipaksakan menjadi model tervalidasi.

Profil memakai nama netral `KMEANS_Pn` dan `DBSCAN_Pn`. ID bukan urutan intensitas dan tidak setara antaralgoritma. Profiling memakai nilai asli dan dukungan data. Tidak ada label risiko rendah/sedang/tinggi otomatis. PCA hanya proyeksi visual; model dipelajari pada lima fitur standar.

## 6. Evaluasi dan kestabilan

Silhouette lebih tinggi dan DBI lebih rendah membantu menilai struktur geometris, tetapi bukan accuracy. Noise dikeluarkan dari metrik jarak dan proporsinya wajib dilaporkan. Tabel perbandingan mencakup hasil seluruh training serta **subset identik yang assigned oleh DBSCAN** untuk kedua algoritma. Pemilihan parameter memakai rata-rata tiga seed; tabel evaluasi ringkas memakai seed 42 yang sama. Nilai keduanya dapat sedikit berbeda karena sampling.

ARI membandingkan partisi pada observasi yang sama dan tidak memerlukan pencocokan nomor label. Uji meliputi seed serta k K-Means, eps/min_samples DBSCAN, resolusi spasial, dan threshold magnitudo. Untuk resolusi berbeda, baris grid tidak sama; label dipetakan kembali ke **event ID historis yang sama**. ARI pada event bersama bersifat **event-weighted**, lebih dipengaruhi daerah dengan banyak kejadian, tidak mencakup grid kosong, dan bukan validasi spasial independen. Overlap serta assigned fraction dilaporkan. ARI termasuk noise dapat didominasi noise, sehingga ARI common-assigned juga disediakan. Ini analisis sensitivitas, belum spatial block bootstrap atau interval ketidakpastian inferensial.

Sensitivitas dukungan minimal 1/3/5/10 event per grid-triwulan juga diuji. Model dan scaler di-fit ulang pada subset historis yang memenuhi dukungan, dengan parameter model terpilih tetap. ARI dibandingkan pada baris grid-triwulan identik yang dipertahankan dan coverage dilaporkan. Ini penting karena mean/maksimum magnitudo sama dan proporsi dangkal hanya 0/1 ketika suatu unit hanya berisi satu event. Cluster dapat dipengaruhi diskretisasi ini; Silhouette bagus belum membuktikan profil regional yang kuat.

## 7. Transisi historis

Satu model yang di-fit pada gabungan 2021-2025 menjaga identitas profil. Pergantian profil hanya dihitung antara dua triwulan **berurutan yang sama-sama assigned ke profil**. Noise, nol event, fitur tidak lengkap dan celah periode memutus run. Stability rate = 1 - changes / comparable pairs. Tanpa pasangan layak, stability rate = NaN. Matriks seluruh state disimpan terpisah. Durasi dinyatakan dalam triwulan, bukan hari.

Pergantian label tidak otomatis perubahan tektonik. Periksa perubahan fitur, jumlah kejadian, kelengkapan katalog dan sensitivitas parameter. Peta count pergantian selalu menampilkan denominator pasangan. Matriks probabilitas merupakan proporsi transisi empiris deskriptif, bukan model Markov prediktif yang tervalidasi.

## 8. Testing 2026

K-Means memakai `predict` dari centroid historis. Jarak centroid >kuantil 99% training merupakan flag eksploratif; threshold ini bukan interval kepercayaan dan tidak membuktikan anomali fisik. DBSCAN memakai **ekstensi core-radius**, bukan predict bawaan: tepat satu cluster core dalam eps -> ditugaskan; tidak ada -> `NO_CORE_MATCH`; lebih dari satu -> `AMBIGUOUS_CORE_MATCH`. Model/scaler tidak di-fit ulang di 2026. `NO_CORE_MATCH` bukan otomatis cluster baru atau noise hasil DBSCAN refit.

Q1-Q3 dilaporkan sebagai quarter lengkap dan Q4 parsial berada dalam CSV/row evaluasi terpisah. Perbandingan baseline di testing memakai profil **2025Q4** secara eksplisit, bukan klaim transisi dari quarter tepat sebelumnya. Tidak menghitung accuracy, precision, recall atau confusion matrix terhadap label yang dibuat model sendiri. Tidak ada informasi kapan gempa akan terjadi.

## 9. Materi pengguna dan referensi teknis

PDF yang diberikan berisi 22 slide rancangan 2025, sedangkan teks saran membahas pengembangan 2021-2025 dan holdout 2026. Implementasi mengikuti klarifikasi pengguna: **2021-2025 training, 2026 testing**. Saran dan perintah yang tertulis dalam lampiran diperlakukan sebagai materi tinjauan; tidak otomatis dijalankan sebagai instruksi tambahan.

- [EMSC FDSN Event service: parameter, format, limit, lisensi](https://www.seismicportal.eu/fdsn-wsevent.html)
- [Scikit-learn clustering: K-Means, DBSCAN dan evaluasi](https://scikit-learn.org/stable/modules/clustering.html)
- [Scikit-learn DBSCAN API](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html)
- [PyProj Transformer](https://pyproj4.github.io/pyproj/stable/api/transformer.html)
- [ISC/IASPEI Seismic Format: kode tipe event](https://www.isc.ac.uk/standards/isf/)

Tanggal peninjauan: 26 Oktober 2026. Referensi teknis mendukung penggunaan API/algoritma. Bukti research gap jurnal masih perlu ditulis dari telaah literatur; proyek ini tidak mengklaim novelty hanya dari dashboard atau pilihan algoritma.


<a id="s4"></a>
## 4. Lokasi input dan keluaran

`data/raw`: JSON asli EMSC dan manifest. `data/processed`: panel dan fitur. `models/kmeans` serta `models/dbscan`: model terpisah. `reports`: tabel, grafik, transisi, sensitivitas, testing. `maps`: peta polygon. Notebook asli di folder induk dipertahankan.

In [4]:
display(Markdown((ROOT / "README.md").read_text(encoding="utf-8")))

# Penelitian aktivitas seismik: training 2021-2025, testing 2026

**Mulai dari [00_panduan.ipynb](notebooks/00_panduan.ipynb).** Notebook lama `../ml_20206 (1).ipynb` tidak diubah. Seluruh pekerjaan baru berada dalam folder ini.

Hasil aktual dan batas interpretasinya dicatat pada [ringkasan hasil eksekusi](reports/ringkasan_hasil.md).

Untuk membaca legenda dan menyiapkan presentasi, buka [cara menjelaskan hasil kepada dosen](docs/cara_menjelaskan_hasil.md). Nama kelompok di peta adalah ringkasan statistik setelah clustering; kelompok tetap dibentuk algoritma. Angka legenda merangkum training, sedangkan popup menunjukkan data kotak pada periode pilihan.

## Notebook dan urutan menjalankan

| No. | Notebook | Keluaran utama |
|---|---|---|
| 00 | [Panduan](notebooks/00_panduan.ipynb) | Pemeriksaan kernel, pustaka, dan metodologi |
| 01 | [Data collection](notebooks/01_data_collection/01_data_collection.ipynb) | Cache JSON bulanan dan manifest EMSC |
| 02 | [Data preprocessing](notebooks/02_data_preprocessing/02_data_preprocessing.ipynb) | Panel grid-triwulan, fitur, scaler historis |
| 03 | [EDA](notebooks/03_eda/03_eda.ipynb) | Audit katalog, distribusi, korelasi |
| 04 | [K-Means](notebooks/04_kmeans/04_kmeans.ipynb) | Seleksi k, centroid, label, profil |
| 05 | [DBSCAN](notebooks/05_dbscan/05_dbscan.ipynb) | k-distance, pencarian eps/min_samples, noise |
| 06 | [Model evaluation](notebooks/06_model_evaluation/06_model_evaluation.ipynb) | Perbandingan cakupan dan subset identik |
| 07 | [Temporal analysis](notebooks/07_temporal_analysis/07_temporal_analysis.ipynb) | Riwayat, transisi, durasi, kestabilan per grid |
| 08 | [Stability](notebooks/08_stability/08_stability.ipynb) | ARI parameter, grid, magnitudo, kebijakan seleksi |
| 09 | [Testing 2026](notebooks/09_testing_2026/09_testing_2026.ipynb) | Hasil Q1-Q3 dan Q4 parsial, drift, core matching |
| 10 | [GIS mapping](notebooks/10_gis_mapping/10_gis_mapping.ipynb) | Peta polygon dengan pemilih quarter dan peta perubahan |

Setiap notebook memuat penjelasan, kode proses yang terlihat, section bernomor, daftar isi, tautan navigasi, dan pemeriksaan hasil. Gunakan Outline pada VS Code atau Table of Contents Jupyter untuk melompat/melipat section. Penomoran folder menjaga urutan di Explorer. Jalankan **Run All** sesuai nomor; tiap notebook memakai kernel baru dan membaca hasil tahap sebelumnya dari disk. Memindahkan urutan cell dapat memutus dependensi, sehingga pengurutan dimaksudkan sebagai navigasi tahapan, bukan eksekusi bebas urutan.

## Kernel Python 3.12.11

Interpreter pilihan pengguna: `C:\Users\yudhi\.local\bin\python3.12.exe`, versi 3.12.11 dari uv. Semua dependensi proyek diinstal di interpreter tersebut, termasuk ipykernel. Kernel terdaftar sebagai **Python 3.12.11 (Seismic Research)**.

Di VS Code: buka notebook, klik **Select Kernel**, pilih **Python 3.12.11 (Seismic Research)**. Jika daftar belum berubah, jalankan **Developer: Reload Window**. Jika Pylance masih memakai interpreter lain, **Python: Select Interpreter** lalu pilih `.local\bin\python3.12.exe`. Jangan memilih Python312 versi 3.12.6 yang merupakan instalasi lain. Notebook mencetak `sys.executable` agar pilihan bisa diperiksa.

Untuk mengulang setup di mesin ini:

```powershell
.\setup_kernel.ps1
```

Python tersebut dikelola uv, sehingga instalasi global yang secara eksplisit diminta menggunakan opsi `--break-system-packages`. `requirements.txt` berisi dependensi proyek; `requirements-lock.txt` mencatat versi environment saat diverifikasi. `reports/validation/environment.json` menyimpan hasil import dan interpreter yang benar-benar dipakai.

## Struktur hasil

```text
notebooks/              notebook utama per tahap/metode
data/raw/monthly/       respons JSON asli, bisa dipakai offline
data/processed/         events bersih, panel, scaled features, scaler
models/kmeans/          model, search, profil, label training
models/dbscan/          model, search, profil, label training
reports/evaluation/     audit katalog dan perbandingan model
reports/temporal/       history, transisi, durasi per metode
reports/stability/      ARI dan sensitivitas kebijakan
reports/testing_2026/   test lengkap dan quarter parsial terpisah
reports/figures/        grafik EDA dan clustering
reports/validation/     bukti eksekusi notebook, environment, dan pengujian
maps/                   HTML interaktif serta geometri GeoJSON
src/                    fungsi pendukung dan kontrak data bersama
scripts/                reproduksi notebook dan pemeriksaan
tests/                  pengujian integritas metodologi
docs/                   keputusan metodologi dan catatan sumber
```

## Konfigurasi dan cara membaca hasil

Pengaturan ada di `config.json`. Training 2021-2025, testing sebelum 2026-10-26 00:00 UTC. **2026Q4 belum lengkap** dan dipisah dari Q1-Q3. Grid utama 50 km pada proyeksi LAEA; sensitivitas 25/100 km. Ambang M>=3.0 masih eksploratif. Jangan menyamakan batas unduhan lengkap dengan katalog yang pasti lengkap secara seismologis.

Jika konfigurasi berubah, jalankan ulang notebook 01 bila cakupan unduhan berubah, lalu 02 sampai 10. Hash konfigurasi/data mendeteksi pemakaian preprocessing/model lama. Peta dan tabel ringkasan harus dihasilkan ulang setelah tahap model berubah. Jangan mencampurkan file hasil dari dua konfigurasi.

**Baca [metodologi dan batas interpretasi](docs/metodologi.md)** sebelum menggunakan hasil dalam laporan. Profil aktivitas bukan peta risiko. Tidak ada label ground truth atau accuracy. DBSCAN pada 2026 menggunakan aturan core-radius tambahan yang menangani kondisi ambigu. Pergantian profil perlu ditinjau bersama dukungan data dan hasil sensitivitas.

## Reproduksi dan verifikasi

```powershell
& "$env:USERPROFILE\.local\bin\python3.12.exe" -m pytest -q
& "$env:USERPROFILE\.local\bin\python3.12.exe" scripts/execute_notebooks.py
```

Skrip kedua menjalankan semua notebook pada kernel yang terdaftar dan menyimpan output di dalam notebook. Data collection menggunakan cache; menghapus/mengganti cache adalah keputusan eksplisit saat memperbarui snapshot katalog. Setiap HTML peta menyertakan Leaflet dan daratan Natural Earth, sehingga bisa dibuka tanpa internet serta tanpa permintaan tile OpenStreetMap. Daratan hanya konteks visual; hasil clustering tetap sama. Sumber dan lisensi ada di `assets/maps/sources.json` dan `LEAFLET_LICENSE.txt`. Jalankan `scripts/prepare_map_assets.py` hanya jika aset lokal hilang; tahap persiapan itu memerlukan internet.


## Referensi dan batas interpretasi

[Dokumentasi EMSC](https://www.seismicportal.eu/fdsn-wsevent.html), [clustering scikit-learn](https://scikit-learn.org/stable/modules/clustering.html), dan docs/metodologi.md. Hasil adalah profil aktivitas katalog dalam bounding box dan periode yang dipilih. Interpretasi risiko, prediksi kejadian gempa, serta klaim perubahan fisik memerlukan bukti tambahan.